# CNN Sentiment Classifier — Interactive Demo

Load the saved checkpoint and test on new text inputs.

In [1]:
import re
import torch
import emoji
from text_cnn import TextCNN

CHECKPOINT_PATH = "./checkpoints/cnn_best.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 40

## 1. Preprocessing

Must match `preprocess_v2` exactly: lowercase, expand contractions, keep stopwords, no lemmatization.

In [2]:
CONTRACTIONS = {
    "can't": "can not", "won't": "will not", "n't": " not",
    "'re": " are", "'s": " is", "'d": " would",
    "'ll": " will", "'ve": " have", "'m": " am",
}
_CONTRACTION_RE = re.compile(
    "(" + "|".join(re.escape(k) for k in CONTRACTIONS) + ")",
    flags=re.IGNORECASE,
)


def preprocess_v2(text: str) -> str:
    text = re.sub(r"<.*?>", " ", text)
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = _CONTRACTION_RE.sub(lambda m: CONTRACTIONS[m.group(0).lower()], text)
    text = emoji.demojize(text).replace(":", " ")
    text = re.sub(r"[^a-z\s!?']", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


print(preprocess_v2("I can't believe how TERRIBLE this movie was!!! @user http://link.com"))
print(preprocess_v2("Absolutely loved it! Best experience ever :)"))

i can not believe how terrible this movie was!!!
absolutely loved it! best experience ever


## 2. Load Checkpoint

The checkpoint contains the model weights, architecture config, and the vocabulary — so we don't need the original training data.

In [3]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)

model_config = checkpoint["model_config"]
vocab_stoi = checkpoint["vocab_stoi"]
vocab_itos = checkpoint["vocab_itos"]

print(f"Model config : {model_config}")
print(f"Vocab size   : {len(vocab_stoi)}")
print(f"Best val F1  : {checkpoint['best_val_f1']:.4f}")

Model config : {'vocab_size': 20000, 'embed_dim': 128, 'num_filters': 100, 'kernel_sizes': (3, 4, 5), 'dropout': 0.5, 'pad_idx': 0}
Vocab size   : 20000
Best val F1  : 0.8212


In [4]:
model = TextCNN(**model_config).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

num_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded — {num_params:,} parameters")

Model loaded — 2,714,201 parameters


## 3. Inference Helpers

Encode text using the saved vocabulary, pad/truncate to `MAX_LEN`, and run through the model.

In [5]:
PAD_IDX = vocab_stoi["<PAD>"]
UNK_IDX = vocab_stoi["<UNK>"]


def encode(text: str) -> list[int]:
    tokens = text.split()
    return [vocab_stoi.get(t, UNK_IDX) for t in tokens]


def pad_or_truncate(ids: list[int]) -> list[int]:
    if len(ids) < MAX_LEN:
        return ids + [PAD_IDX] * (MAX_LEN - len(ids))
    return ids[:MAX_LEN]


@torch.no_grad()
def predict(raw_text: str) -> dict:
    cleaned = preprocess_v2(raw_text)
    ids = pad_or_truncate(encode(cleaned))
    input_tensor = torch.tensor([ids], dtype=torch.long, device=DEVICE)

    logit = model(input_tensor).item()
    prob = torch.sigmoid(torch.tensor(logit)).item()
    label = "POSITIVE" if prob >= 0.5 else "NEGATIVE"

    return {
        "raw_text": raw_text,
        "cleaned_text": cleaned,
        "tokens": cleaned.split(),
        "num_tokens": len(cleaned.split()),
        "prediction": label,
    }

## 4. Test on New Data Points

In [6]:
test_tweets = [
    "I absolutely love this product! Best purchase I've ever made.",
    "Worst experience of my life. Totally disappointed and won't come back.",
    "The weather is okay today, nothing special really.",
]

for tweet in test_tweets:
    result = predict(tweet)
    print("=" * 70)
    print(f"Input   : {result['raw_text']}")
    print(f"Cleaned : {result['cleaned_text']}")
    print(f"Tokens  : {result['num_tokens']} words")
    print(f"Label   : {result['prediction']}")
    print()

Input   : I absolutely love this product! Best purchase I've ever made.
Cleaned : i absolutely love this product! best purchase i have ever made
Tokens  : 11 words
Label   : POSITIVE

Input   : Worst experience of my life. Totally disappointed and won't come back.
Cleaned : worst experience of my life totally disappointed and will not come back
Tokens  : 12 words
Label   : NEGATIVE

Input   : The weather is okay today, nothing special really.
Cleaned : the weather is okay today nothing special really
Tokens  : 8 words
Label   : NEGATIVE

